In [12]:
import requests

# api_key = 'ZZ7ZN51J8CYOT047'
api_key = 'JCFRQP4VNPA72GWZ'
keywords = 'Unilever'
url = f'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords={keywords}&apikey={api_key}'
r = requests.get(url)
print(r.json())

{'bestMatches': [{'1. symbol': 'UNLVF', '2. name': 'Unilever NV', '3. type': 'Equity', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.8421'}, {'1. symbol': 'UL', '2. name': 'Unilever plc', '3. type': 'Equity', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.8000'}, {'1. symbol': 'UNLYF', '2. name': 'Unilever plc', '3. type': 'Equity', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.8000'}, {'1. symbol': 'UNVB.DEX', '2. name': 'Unilever Plc', '3. type': 'Equity', '4. region': 'XETRA', '5. marketOpen': '08:00', '6. marketClose': '20:00', '7. timezone': 'UTC+02', '8. currency': 'EUR', '9. matchScore': '0.8000'}, {'1. symbol': 'UNVB.FRK', '2. name': 'Unilever Plc', '3. type': 'Equity', '4

In [14]:
import pandas as pd
import numpy as np
import time

In [13]:
# GET ANNUALIZED VOLATILITY
import pandas as pd
import numpy as np
import time
symbols = ['ULVR.LON']

for symbol in symbols:
    print(f"Processing {symbol}...")

    url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )

    df = pd.read_csv(url, parse_dates=['timestamp'])
    df = df.sort_values('timestamp')  # oldest to newest

    # Use last 252 trading days for 1-year volatility
    df_last_year = df.tail(252)
    prices = df_last_year['close']
    
    # Convert GBX to GBP if needed
    prices_gbp = prices / 100.0

    # Calculate log returns
    returns = np.log(prices_gbp / prices_gbp.shift(1)).dropna()

    # Daily volatility
    daily_vol = returns.std()

    # Annualized volatility
    annual_vol = daily_vol * np.sqrt(252)

    print(f"{symbol} 1Y annualized volatility: {annual_vol:.2%} (GBP-based)")
    time.sleep(3)


Processing ULVR.LON...
ULVR.LON 1Y annualized volatility: 17.24% (GBP-based)


In [ ]:

# GET ANNUALIZED VOLATILITY IN CHF
%reset -f
symbols = ['SWDA.LON']

for symbol in symbols:
    print(f"Processing {symbol}...")

    url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )
    # get the asset price data
    asset_df = pd.read_csv(url, parse_dates=['timestamp'])
    asset_df = asset_df.rename(columns={'close': 'asset_close'})
    asset_df = asset_df.set_index('timestamp')  # oldest to newest
    # Use last 252 trading days for 1-year volatility
    df_last_year = asset_df.tail(252)


    fx_url = (
    f'https://www.alphavantage.co/query?function=FX_DAILY'
    f'&from_symbol=GBP&to_symbol=CHF&apikey={api_key}&outputsize=full&datatype=csv'
    )
    fx_df = pd.read_csv(fx_url, parse_dates=['timestamp'])
    fx_df = fx_df.rename(columns={'close': 'fx_rate'})
    fx_df = fx_df.sort_values('timestamp')
    fx_df = fx_df.set_index('timestamp')['close']  # Use 'close' as FX rate

    # Now merge with your asset price series on date and multiply:
    df = asset_df.join(fx_df, how='inner')
    df['close_chf'] = df['asset_close'] * df['fx_rate']  # Convert to CHF

    # Continue as before, but on 'close_chf'
    prices = df['close_chf']
    returns = np.log(prices / prices.shift(1)).dropna()


    # Daily volatility
    daily_vol = returns.std()

    # Annualized volatility
    annual_vol = daily_vol * np.sqrt(252)

    print(f"{symbol} 1Y annualized volatility: {annual_vol:.2%} (GHF-based)")
    # time.sleep(3)


Processing SWDA.LON...
SWDA.LON 1Y annualized volatility: 14.57% (GHF-based)


In [8]:
%reset -f
%whos

Interactive namespace is empty.


In [32]:

symbol = ['ULVR.LON']

print(f"Processing {symbol}...")
url = (
    f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
    f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
)
response = requests.get(url)
print(response.text[:500])  
# Print first 500 characters to see what is returned

# Try to parse only if 'timestamp' is in the header
if 'timestamp' in response.text.split('\n')[0]:
    df = pd.read_csv(StringIO(response.text), parse_dates=['timestamp'])
    df = df.sort_values('timestamp')  # oldest to newest

    # get price on specific date
    specific_date = '2024-10-08'
    if specific_date in df['timestamp'].values:
        price_on_date = df.loc[df['timestamp'] == specific_date, 'close'].values[0]
        print(f"Price on {specific_date}: {price_on_date}")
    else:
        print(f"No data available for {specific_date}")

    df_last_year = df.tail(252)
    prices = df_last_year['close']

    # Convert GBX to GBP if needed
    prices_gbp = prices / 100.0
else:
    print("Response does not contain a valid CSV with 'timestamp'. Check API key, symbol, or rate limits.")


# df_last_year = df.tail(252)
# prices = df_last_year['close']

# # Convert GBX to GBP if needed
# prices_gbp = prices / 100.0

Processing ['ULVR.LON']...
{
    "Error Message": "Invalid API call. Please retry or visit the documentation (https://www.alphavantage.co/documentation/) for TIME_SERIES_DAILY."
}
Response does not contain a valid CSV with 'timestamp'. Check API key, symbol, or rate limits.
